# Explorador de Murmullos BSD

## Experimento Amateur: Clasificación de Rangos de Curvas Elípticas mediante Murmuraciones

Este notebook implementa un experimento para comprobar la validez de la clasificación de rangos de curvas elípticas mediante el fenómeno de las murmuraciones, utilizando datos reales de LMFDB (L-functions and Modular Forms Database).

**Autor:** Experimento amateur
**Fecha:** Marzo 2026
**Contexto:** Conjetura de Birch y Swinnerton-Dyer (Problema del Milenio)

## 1. Instalación de Dependencias

Instalamos las librerías necesarias para el análisis.

In [ ]:
!pip install requests pandas numpy matplotlib seaborn scipy

## 2. Importación de Librerías

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import List, Dict
import json
import time

# Configuración de visualización
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Librerías importadas correctamente')

## 3. Conexión a LMFDB

Descargamos datos reales de curvas elípticas desde la base de datos LMFDB.

In [ ]:
def obtener_curvas_lmfdb(conductor_max=1000, limit=500):
    """
    Descarga curvas elípticas de LMFDB.
    
    Parámetros:
    - conductor_max: conductor máximo
    - limit: número máximo de curvas
    """
    base_url = 'https://www.lmfdb.org/api/ec_curvedata'
    
    params = {
        'conductor': {'$lte': conductor_max},
        '_format': 'json',
        '_limit': limit
    }
    
    try:
        print(f'Descargando curvas con conductor <= {conductor_max}...')
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'data' in data:
            df = pd.DataFrame(data['data'])
            print(f'Descargadas {len(df)} curvas exitosamente')
            return df
        else:
            print('No se encontraron datos')
            return pd.DataFrame()
    except Exception as e:
        print(f'Error al descargar datos: {e}')
        return pd.DataFrame()

# Descargar datos
df_curvas = obtener_curvas_lmfdb(conductor_max=1000, limit=500)
df_curvas.head()

## 4. Análisis Exploratorio de Datos

In [ ]:
# Información básica del dataset
print('Dimensiones del dataset:', df_curvas.shape)
print('\nColumnas disponibles:')
print(df_curvas.columns.tolist())

# Distribución de rangos
if 'rank' in df_curvas.columns:
    print('\nDistribución de rangos:')
    print(df_curvas['rank'].value_counts().sort_index())
    
    # Visualización
    plt.figure(figsize=(10, 5))
    df_curvas['rank'].value_counts().sort_index().plot(kind='bar')
    plt.title('Distribución de Rangos de Curvas Elípticas')
    plt.xlabel('Rango')
    plt.ylabel('Frecuencia')
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.show()

## 5. Cálculo de Murmuraciones

Implementamos el cálculo de los patrones de murmuración para clasificar curvas por rango.

In [ ]:
def calcular_murmuración(conductor, coeficientes):
    """
    Calcula el patrón de murmuración para una curva elíptica.
    
    La murmuración es un fenómeno observado en el espacio de parámetros
    de las curvas elípticas que puede correlacionarse con su rango.
    """
    if not isinstance(coeficientes, list) or len(coeficientes) < 2:
        return 0
    
    # Fórmula simplificada de murmuración
    # Basada en la relación entre conductor y coeficientes
    a1, a2 = coeficientes[0], coeficientes[1] if len(coeficientes) > 1 else 0
    
    # Cálculo del índice de murmuración
    theta = np.arctan2(a2, conductor) if conductor != 0 else 0
    r = np.sqrt(a1**2 + a2**2) / (conductor + 1)
    
    murmuración = r * np.cos(theta)
    
    return murmuración

# Aplicar cálculo a todas las curvas
if 'conductor' in df_curvas.columns and 'ainvs' in df_curvas.columns:
    df_curvas['murmuración'] = df_curvas.apply(
        lambda row: calcular_murmuración(row['conductor'], row['ainvs']),
        axis=1
    )
    print('Murmuraciones calculadas')
    print(df_curvas[['conductor', 'rank', 'murmuración']].head(10))

## 6. Visualización de Murmuraciones por Rango

In [ ]:
# Gráfico de dispersión: Murmuración vs Conductor, coloreado por Rango
if 'murmuración' in df_curvas.columns and 'rank' in df_curvas.columns:
    plt.figure(figsize=(14, 8))
    
    # Filtrar valores válidos
    df_plot = df_curvas[df_curvas['murmuración'].notna()].copy()
    
    # Crear gráfico
    scatter = plt.scatter(
        df_plot['conductor'],
        df_plot['murmuración'],
        c=df_plot['rank'],
        cmap='viridis',
        alpha=0.6,
        s=50
    )
    
    plt.colorbar(scatter, label='Rango')
    plt.xlabel('Conductor')
    plt.ylabel('Patrón de Murmuración')
    plt.title('Patrones de Murmuración de Curvas Elípticas por Rango')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Distribución de murmuraciones por rango
    plt.figure(figsize=(12, 6))
    for rank in sorted(df_plot['rank'].unique()):
        data = df_plot[df_plot['rank'] == rank]['murmuración']
        plt.hist(data, alpha=0.5, bins=30, label=f'Rango {int(rank)}')
    
    plt.xlabel('Patrón de Murmuración')
    plt.ylabel('Frecuencia')
    plt.title('Distribución de Murmuraciones por Rango')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Análisis Estadístico: Correlación entre Murmuración y Rango

In [ ]:
if 'murmuración' in df_curvas.columns and 'rank' in df_curvas.columns:
    # Eliminar valores NaN
    df_clean = df_curvas[['rank', 'murmuración', 'conductor']].dropna()
    
    # Correlación de Pearson
    corr_pearson, p_value_pearson = stats.pearsonr(
        df_clean['murmuración'],
        df_clean['rank']
    )
    
    # Correlación de Spearman
    corr_spearman, p_value_spearman = stats.spearmanr(
        df_clean['murmuración'],
        df_clean['rank']
    )
    
    print('=== ANÁLISIS ESTADÍSTICO ===')
    print(f'\nNúmero de curvas analizadas: {len(df_clean)}')
    print(f'\nCorrelación de Pearson: {corr_pearson:.4f}')
    print(f'P-value (Pearson): {p_value_pearson:.4e}')
    print(f'\nCorrelación de Spearman: {corr_spearman:.4f}')
    print(f'P-value (Spearman): {p_value_spearman:.4e}')
    
    # Interpretación
    if p_value_pearson < 0.05:
        print('\n✅ Existe una correlación estadísticamente significativa')
    else:
        print('\n❌ No se encuentra correlación significativa')
    
    # Estadísticas descriptivas por rango
    print('\n=== ESTADÍSTICAS POR RANGO ===')
    stats_por_rango = df_clean.groupby('rank')['murmuración'].describe()
    print(stats_por_rango)

## 8. Modelo de Clasificación de Rango

Intentamos predecir el rango de una curva elíptica usando su patrón de murmuración.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

if 'murmuración' in df_curvas.columns and 'rank' in df_curvas.columns:
    # Preparar datos
    df_model = df_curvas[['conductor', 'murmuración', 'rank']].dropna()
    
    X = df_model[['conductor', 'murmuración']]
    y = df_model['rank']
    
    # Dividir en entrenamiento y prueba
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # Entrenar modelo
    modelo = RandomForestClassifier(n_estimators=100, random_state=42)
    modelo.fit(X_train, y_train)
    
    # Predecir
    y_pred = modelo.predict(X_test)
    
    # Evaluación
    print('=== RESULTADOS DEL MODELO DE CLASIFICACIÓN ===')
    print(f'\nPrecisión (Accuracy): {accuracy_score(y_test, y_pred):.4f}')
    print('\nReporte de Clasificación:')
    print(classification_report(y_test, y_pred))
    
    # Matriz de confusión
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Matriz de Confusión: Clasificación de Rango')
    plt.ylabel('Rango Real')
    plt.xlabel('Rango Predicho')
    plt.tight_layout()
    plt.show()
    
    # Importancia de características
    importancias = pd.DataFrame({
        'Característica': X.columns,
        'Importancia': modelo.feature_importances_
    }).sort_values('Importancia', ascending=False)
    
    print('\nImportancia de Características:')
    print(importancias)

## 9. Conclusiones

Este experimento amateur explora la relación entre los patrones de murmuración y el rango de curvas elípticas.

### Hallazgos Principales:

1. **Descarga de Datos Reales**: Se obtuvieron curvas elípticas reales de la base de datos LMFDB.

2. **Cálculo de Murmuraciones**: Se implementó una fórmula simplificada basada en la relación entre el conductor y los coeficientes de las curvas.

3. **Análisis Estadístico**: Se calcularon correlaciones entre los patrones de murmuración y el rango de las curvas.

4. **Modelo de Clasificación**: Se entrenó un Random Forest para predecir el rango basado en murmuraciones.

### Limitaciones:

- Este es un experimento amateur con fines educativos.
- La fórmula de murmuración es simplificada y puede no capturar toda la complejidad del fenómeno.
- Se requiere validación con expertos en teoría de números.

### Conexión con BSD:

La Conjetura de Birch y Swinnerton-Dyer es uno de los 7 Problemas del Milenio. Este experimento explora patrones que podrían ayudar a entender la distribución de rangos, un aspecto crucial de la conjetura.

### Próximos Pasos:

- Ampliar el dataset con más curvas
- Refinar la fórmula de murmuración
- Consultar con matemáticos profesionales
- Explorar otras características de las curvas

---

**Repositorio**: [BSD-Murmurations-Explorer](https://github.com/mcarrionsiete/BSD-Murmurations-Explorer)

**Licencia**: MIT

*Experimento realizado con fines educativos y de investigación amateur.*